In [0]:
-- ============================================
-- SILVER 层: 数据清洗和转换
-- ============================================

-- 1. 处理时间维度 
insert overwrite aws3.silver.dim_date
WITH date_series AS (
    SELECT explode(sequence(
        to_date('2020-01-01'), 
        to_date('2030-12-31'), 
        interval 1 day
    )) as date
)
SELECT 
    -- 代理键
    CAST(date_format(date, 'yyyyMMdd') AS INT) as date_key,
    
    -- 日期属性
    date as full_date,
    YEAR(date) as year,
    QUARTER(date) as quarter,
    MONTH(date) as month,
    DAY(date) as day,
    
    -- 周信息
    WEEKOFYEAR(date) as week_of_year,
    DAYOFWEEK(date) as day_of_week,
    
    -- 业务标记
    CASE WHEN DAYOFWEEK(date) IN (1,7) THEN 1 ELSE 0 END as is_weekend,
    CASE WHEN MONTH(date) = 12 AND DAY(date) = 25 THEN 1 ELSE 0 END as is_christmas,
    CURRENT_TIMESTAMP() as load_time
    
FROM date_series;